In [2]:
pwd

'C:\\Users\\ameyn\\FULL SPECTRA CLASSIFICATION\\Code to separate bacteria signals from other signals'

In [1]:

import os
import numpy as np
import pandas as pd
import joblib

from preprocessing_utils import (
    baseline_correction,
    snv_normalization,
    interpolate_to_reference,
    vector_normalization
)

# ---------- Step 1: Define Common Reference Axis ----------
common_start = 600.3
common_end = 1848.2
n_points = 500
reference_axis = np.linspace(common_start, common_end, n_points)

# ---------- Step 2: Define dataset paths ----------
dataset1_paths = [r"D:\disk D Downloads\DATA FROM BECHLING_GOLD_CHIP_Ecoli_diff_strains\Ecoli_Different _strains\ATCC25922 (non-pathogenic E. coli)\trim without BL\R061_Au251218_785nm_Edge_600_750nm_x50_VIS_DF_100__1_10_s_ATCC25922.csv",
                  r"D:\disk D Downloads\DATA FROM BECHLING_GOLD_CHIP_Ecoli_diff_strains\Ecoli_Different _strains\ATCC25922 (non-pathogenic E. coli)\trim without BL\R062_Au251218_785nm_Edge_600_750nm_x20_VIS_DF_100__1_15_s_ATCC25922.csv",
                  r"D:\disk D Downloads\DATA FROM BECHLING_GOLD_CHIP_Ecoli_diff_strains\Ecoli_Different _strains\E. coli O103H2 NIST0058 (STEC E. coli)\trim without BL\R230_Au_260120_785nm_Edge_600_750nm_x50_VIS_DF_25__1_10_s_O103_H2.csv",
                  r"D:\disk D Downloads\DATA FROM BECHLING_GOLD_CHIP_Ecoli_diff_strains\Ecoli_Different _strains\E. coli O121H19 NIST0059 (STEC E. coli)\trimmed_without BL\R228_Au_260120_785nm_Edge_600_750nm_x50_VIS_DF_25__1_10_s_O121_H19.csv",
                  r"D:\disk D Downloads\DATA FROM BECHLING_GOLD_CHIP_Ecoli_diff_strains\Ecoli_Different _strains\E. coli O157H7 Pathotrak EC1 (Kanmycin resistant STEC E. coli)\trim_without_BL\R232_Au_260120_785nm_Edge_600_750nm_x50_VIS_DF_25__1_10_s_O157_H7.csv",

                  r"D:\disk D Downloads\DATA FROM BENCHLING_GOLD_CHIP_SALMONELLA\trim_without_BL\Au251210_785nm_Edge_600_750nm_x20_VIS_DF_100__1_10_s_ATCC700408_Area_1.csv",
                  r"D:\disk D Downloads\DATA FROM BENCHLING_GOLD_CHIP_SALMONELLA\trim_without_BL\Au251210_785nm_Edge_600_750nm_x50_VIS_DF_100__1_10_s_ATCC700408_Area_1.csv"
]

##OLD DATA:
dataset2_paths = [r"D:\disk D Downloads\ONE CLASS SVM AND NC\interpolating Au and Ag chips\NEW-Recreating reference Ag and Au datasets-SNR\Au chips\20250410 Data Dump\Au062 _100 %_x100_VIS_DF_1200 (750nm)_638nm_Edge_1_10 s_A1 SE in H2O_1.csv",
                  r"D:\disk D Downloads\ONE CLASS SVM AND NC\interpolating Au and Ag chips\NEW-Recreating reference Ag and Au datasets-SNR\Au chips\20250410 Data Dump\Au062 _100 %_x100_VIS_DF_1200 (750nm)_638nm_Edge_1_10 s_A1 SE in H2O_2.csv",
                  r"D:\disk D Downloads\ONE CLASS SVM AND NC\interpolating Au and Ag chips\NEW-Recreating reference Ag and Au datasets-SNR\Au chips\20250506 RS0183 Bacteria on Au\Au088_100 %_x10_VIS_DF_600 (750nm)_785nm_Edge_1_45 s_A1 EC.csv",
                  r"D:\disk D Downloads\ONE CLASS SVM AND NC\interpolating Au and Ag chips\NEW-Recreating reference Ag and Au datasets-SNR\Au chips\20250506 RS0183 Bacteria on Au\Au088_100 %_x10_VIS_DF_600 (750nm)_785nm_Edge_1_90 s_A1 EC.csv",
                  r"D:\disk D Downloads\ONE CLASS SVM AND NC\interpolating Au and Ag chips\NEW-Recreating reference Ag and Au datasets-SNR\Au chips\20250506 RS0183 Bacteria on Au\Au088_100 %_x10_VIS_DF_600 (750nm)_785nm_Edge_1_90 s_B2 LM 2.csv",
                  r"D:\disk D Downloads\ONE CLASS SVM AND NC\interpolating Au and Ag chips\NEW-Recreating reference Ag and Au datasets-SNR\Au chips\20250506 RS0183 Bacteria on Au\Au088_100 %_x10_VIS_DF_600 (750nm)_785nm_Edge_1_90 s_B2 LM.csv",
                  r"D:\disk D Downloads\ONE CLASS SVM AND NC\interpolating Au and Ag chips\NEW-Recreating reference Ag and Au datasets-SNR\Au chips\S81_Au_More of Au_data\S81 Au A2 S Typhimurium 10percent 90s 100xOBJ RS0051.csv",
                  r"D:\disk D Downloads\ONE CLASS SVM AND NC\interpolating Au and Ag chips\NEW-Recreating reference Ag and Au datasets-SNR\Au chips\S81_Au_More of Au_data\S81 Au A2 S Typhimurium 100percent 10s 1acc 10xOBJ RS0049.csv",
                  r"D:\disk D Downloads\ONE CLASS SVM AND NC\interpolating Au and Ag chips\NEW-Recreating reference Ag and Au datasets-SNR\Au chips\S81_Au_More of Au_data\S81 Au A2 S Typhimurium 100percent 10s 1um 50xOBJ RS0050.csv",
                  r"D:\disk D Downloads\ONE CLASS SVM AND NC\interpolating Au and Ag chips\NEW-Recreating reference Ag and Au datasets-SNR\Au chips\S81_Au_More of Au_data\S81 Au A2 S Typhimurium 100percent 10s 100xOBJ RS0051.csv"
]


dataset3_paths = [r"D:\disk D Downloads\O157 collected on different dates\Sorted_according_to_date\13_May\Automated_by_claude\04_clean_trimmed_raw\R691_Au_260513_785nm_Edge_600_750nm_x50_VIS_DF_100__1_10_s_O157H7_B1.csv",
                  r"D:\disk D Downloads\O157 collected on different dates\Sorted_according_to_date\13_May\Automated_by_claude\04_clean_trimmed_raw\R692_Au_260513_785nm_Edge_600_750nm_x50_VIS_DF_100__1_10_s_O157H7_B2.csv",
                  r"D:\disk D Downloads\O157 collected on different dates\Sorted_according_to_date\13_May\Automated_by_claude\04_clean_trimmed_raw\R693_Au_260513_785nm_Edge_600_750nm_x50_VIS_DF_100__1_10_s_O157H7_B3.csv",

                  r"D:\disk D Downloads\O157 collected on different dates\Sorted_according_to_date\14_May\Automated_by_claude\04_clean_trimmed_raw\R694_Au_260514_785nm_Edge_600_750nm_x50_VIS_DF_100__1_10_s_O157H7_B4.csv",

                  r"D:\disk D Downloads\O157 collected on different dates\Sorted_according_to_date\30_April\Automaed_by_claude\04_clean_trimmed_raw\R631_Au_260430_785nm_Edge_600_750nm_x50_VIS_DF_100__1_10_s_O157_H7.csv"
]


all_paths = dataset1_paths + dataset2_paths + dataset3_paths

# ---------- Helper function for NC preprocessing ----------
def preprocess_for_nc(path):
    df = pd.read_csv(path)
    df.columns = df.columns.astype(float)

    corrected = baseline_correction(df)
    snv = snv_normalization(corrected.values)
    interp = interpolate_to_reference(pd.DataFrame(snv, columns=corrected.columns), reference_axis)

    # Add label
    interp["label"] = os.path.basename(path)
    return interp

# ---------- Helper function for HQI preprocessing ----------
def preprocess_for_hqi(path):
    df = pd.read_csv(path)
    df.columns = df.columns.astype(float)

    corrected = baseline_correction(df)
    vnorm = vector_normalization(corrected.values)
    interp = interpolate_to_reference(pd.DataFrame(vnorm, columns=corrected.columns), reference_axis)

    # Add label
    interp["label"] = os.path.basename(path)
    return interp

# ---------- Process all datasets ----------
processed_nc = [preprocess_for_nc(path) for path in all_paths]
processed_hqi = [preprocess_for_hqi(path) for path in all_paths]

# ---------- Combine ----------
combined_data_nc = pd.concat(processed_nc, ignore_index=True)
combined_data_hqi = pd.concat(processed_hqi, ignore_index=True)

# ---------- Extract labels ----------
dataset_labels = combined_data_nc["label"].tolist()

# ---------- Convert spectral matrices to numpy arrays ----------
combined_nc_array = combined_data_nc.drop(columns=["label"]).values
combined_hqi_array = combined_data_hqi.drop(columns=["label"]).values

print("NC shapes:", combined_nc_array.shape, len(dataset_labels))
print("HQI shapes:", combined_hqi_array.shape)

# ---------- Save everything ----------
joblib.dump({
    "reference_axis": reference_axis,
    "combined_data": combined_nc_array,
    "combined_data_hqi": combined_hqi_array,
    "dataset_labels": dataset_labels
}, "reference_data.pkl")

print("✅ Saved reference_data.pkl with NC + HQI + labels")


NC shapes: (3056, 500) 3056
HQI shapes: (3056, 500)
✅ Saved reference_data.pkl with NC + HQI + labels
